# Staffing CSV — Source Data Analysis

**File:** `SBEMS_Staffing_2022_2025.csv`  
**Purpose:** Document every assumption before this file drives any dashboard metric.  
**Key questions this notebook answers:**
1. What columns does the file actually have?
2. What does each row represent — a shift, a pay period, an hourly entry?
3. How is 'shift' encoded?
4. What is the date range?
5. Is there a join key to the roster file (Badge Number / employee_id)?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

FILE_PATH = Path(r"D:\MullenAnalytics\ClientData\agencies\81d6f2e6-300a-48d0-b7b6-9250937d17fb\raw\ec8927db_SBEMS_Staffing_2022_2025.csv")
ROSTER_PATH = Path(r"D:\MullenAnalytics\ClientData\agencies\81d6f2e6-300a-48d0-b7b6-9250937d17fb\raw\e6532050_User Roster EMT Only for FS 2.3.26.xlsx")

assert FILE_PATH.exists(), f"File not found: {FILE_PATH}"
print(f"File size: {FILE_PATH.stat().st_size / 1_000:.1f} KB")

## 1. Load Raw File

In [ ]:
raw = pd.read_csv(FILE_PATH, low_memory=False)
print(f"Shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print()
print("Columns, dtypes, and null counts:")
for c in raw.columns:
    null_n   = raw[c].isnull().sum()
    null_pct = null_n / len(raw) * 100
    print(f"  {c:<20} dtype={str(raw[c].dtype):<10} nulls={null_n:,} ({null_pct:.1f}%)")

## 2. Random Sample — What does a row actually represent?

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
sample = raw.sample(10, random_state=42)
sample

In [ ]:
# Answer: what does each row represent?
# Look at one employee across multiple rows to confirm row = one shift worked
first_emp = raw['employee_id'].dropna().iloc[0]
print(f"All rows for employee_id = {first_emp}:")
one_emp = raw[raw['employee_id'] == first_emp].sort_values('date')
print(one_emp.to_string())
print(f"\nRows: {len(one_emp)}  — if each row is a different date, row = one shift")

## 3. Column Profiles

In [ ]:
# employee_id
print("=== employee_id ===")
print(f"  Unique values: {raw['employee_id'].nunique()}")
print(f"  Type sample:   {raw['employee_id'].dropna().head(5).tolist()}")
print(f"  Null count:    {raw['employee_id'].isnull().sum()}")
print()

# name
print("=== name ===")
print(f"  Unique names: {raw['name'].nunique()}")
print(f"  Sample:       {raw['name'].dropna().unique()[:5].tolist()}")
print()

# shift — HOW IS IT ENCODED?
print("=== shift ===")
print(f"  Unique values: {sorted(raw['shift'].dropna().unique().tolist())}")
print(f"  Value counts:")
print(raw['shift'].value_counts().to_string())
print()

# date
print("=== date ===")
dates = pd.to_datetime(raw['date'], errors='coerce')
print(f"  Parse failures: {dates.isnull().sum() - raw['date'].isnull().sum()} rows")
print(f"  Min date: {dates.min()}")
print(f"  Max date: {dates.max()}")
print(f"  Date range: {(dates.max() - dates.min()).days} days")
print()

# position
print("=== position ===")
print(raw['position'].value_counts().to_string())
print()

# hours
print("=== hours ===")
hours_n = pd.to_numeric(raw['hours'], errors='coerce')
print(f"  Min: {hours_n.min()}   Max: {hours_n.max()}   Mean: {hours_n.mean():.1f}   Median: {hours_n.median()}")
print(f"  Value counts (top 10):")
print(hours_n.value_counts().head(10).to_string())
print()

# overtime
print("=== overtime ===")
print(raw['overtime'].value_counts().to_string())
print(f"  Overtime flag definition: hours > {raw.loc[raw['overtime']==True,'hours'].min()} on a shift?")
# Check: is overtime = hours > 12?
if 'hours' in raw.columns:
    h = pd.to_numeric(raw['hours'], errors='coerce')
    ot_flag = raw['overtime'].astype(str).str.lower() == 'true'
    print(f"\n  Cross-check — rows flagged OT with hours <= 12: {((h <= 12) & ot_flag).sum()}")
    print(f"  Cross-check — rows NOT flagged OT with hours > 12: {((h > 12) & ~ot_flag).sum()}")

## 4. Required Field Validation (pipeline contract)

In [ ]:
REQUIRED = ["employee_id", "date"]
print("Required field check:")
for f in REQUIRED:
    present = f in raw.columns
    if present:
        null_pct = raw[f].isnull().mean() * 100
        print(f"  {f:<15}: PRESENT   null={null_pct:.1f}%")
    else:
        print(f"  {f:<15}: MISSING")

print()
print("Validator outcome: PASSED" if all(f in raw.columns for f in REQUIRED) else "Validator outcome: FAILED")

## 5. Join Key — Can we link to the roster file?

**Question:** Does `employee_id` in this file match `Badge Number` in the roster?

In [ ]:
roster = pd.read_excel(ROSTER_PATH)
print("Roster columns:", roster.columns.tolist())
print(f"Roster rows: {len(roster)}")

# Normalize badge numbers to same type
staffing_ids = set(raw['employee_id'].dropna().astype(int))
roster_ids   = set(roster['Badge Number'].dropna().astype(int))

matched   = staffing_ids & roster_ids
only_staf = staffing_ids - roster_ids
only_rost = roster_ids   - staffing_ids

print(f"\nStaffing unique employee_ids: {len(staffing_ids)}")
print(f"Roster unique Badge Numbers:  {len(roster_ids)}")
print(f"Matched (in both):            {len(matched)}")
print(f"In staffing only:             {len(only_staf)}")
print(f"In roster only:               {len(only_rost)}")
print()
if only_staf:
    print(f"IDs in staffing but not roster (first 10): {sorted(only_staf)[:10]}")
if only_rost:
    print(f"Badge numbers in roster but not staffing (first 10): {sorted(only_rost)[:10]}")

## 6. Cleaning Steps — one at a time

In [ ]:
df = raw.copy()
print(f"START: {len(df):,} rows")

In [ ]:
# Step 1: Parse date column
df['date'] = pd.to_datetime(df['date'], errors='coerce')
bad_dates = df['date'].isnull().sum()
print(f"Date parse failures: {bad_dates} rows set to NaT")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Span: {(df['date'].max() - df['date'].min()).days} days  ({(df['date'].max() - df['date'].min()).days / 365:.1f} years)")

In [ ]:
# Step 2: Parse hours to numeric
df['hours'] = pd.to_numeric(df['hours'], errors='coerce')
print(f"Hours parse failures: {df['hours'].isnull().sum()}")
print(f"Hours range: {df['hours'].min()} – {df['hours'].max()}")

# Flag implausible shift lengths
implausible = df['hours'] > 24
print(f"Rows with hours > 24 (implausible): {implausible.sum()}")

In [ ]:
# Step 3: Normalize overtime flag to bool
df['overtime'] = df['overtime'].astype(str).str.strip().str.lower().map({'true': True, 'false': False, '1': True, '0': False})
print(f"Overtime: {df['overtime'].sum()} True / {(~df['overtime'].fillna(False)).sum()} False / {df['overtime'].isnull().sum()} null")

## 7. Metric Verification — reproduce every dashboard number

In [ ]:
# METRIC: Unique staff count
# Formula: count of distinct employee_id values
# Includes: all rows (no date filter)
unique_staff = df['employee_id'].nunique()
print(f"Unique staff: {unique_staff}")
print(f"Dashboard shows: check against this number")

In [ ]:
# METRIC: By-shift breakdown
# Formula: count of rows per shift value
# Includes: rows where shift is not null
by_shift = df['shift'].value_counts(dropna=True)
print("By shift (row counts = individual shifts worked):")
print(by_shift.to_string())
print(f"\nTotal rows with shift data: {by_shift.sum():,} of {len(df):,}")

In [ ]:
# METRIC: Overtime rate
# Formula: count(overtime==True) / count(all rows with non-null hours) * 100
# Uses pipeline threshold: shift > 12 hours = OT
ot_by_flag  = df['overtime'].sum() / df['overtime'].notna().sum() * 100
ot_by_hours = (df['hours'] > 12).sum() / df['hours'].notna().sum() * 100
print(f"Overtime rate (from 'overtime' flag):   {ot_by_flag:.1f}%")
print(f"Overtime rate (hours > 12 threshold): {ot_by_hours:.1f}%")
print(f"\nAre they consistent? {'YES' if abs(ot_by_flag - ot_by_hours) < 2 else 'NO — investigate'}")

In [ ]:
# METRIC: Average hours per shift
avg_hrs = df['hours'].mean()
print(f"Average hours per shift (mean): {avg_hrs:.1f}h")
print(f"Median hours per shift:         {df['hours'].median():.1f}h")
print()
# Distribution of shift lengths
print("Shift length distribution:")
print(df['hours'].value_counts().sort_index().to_string())

In [ ]:
# METRIC: Staffing gap days (days with < 2 staff on record)
# Formula: for each date, count unique employee_ids. Flag dates < MIN_STAFFING (2).
MIN_STAFFING = 2
daily_staff = df.groupby(df['date'].dt.date)['employee_id'].nunique()
gap_days    = daily_staff[daily_staff < MIN_STAFFING]
print(f"Total dates with staffing data: {len(daily_staff)}")
print(f"Gap days (< {MIN_STAFFING} staff): {len(gap_days)}")
if len(gap_days) > 0:
    print("Gap day details (first 10):")
    print(gap_days.head(10).to_string())

## 8. Caveats

### 8.1 Row = one shift, not one employee-day
The same employee can appear multiple times on the same date if they worked split shifts  
or if data entry created duplicate records. Verify with the cross-check cell above.

### 8.2 Overtime flag definition
The `overtime` boolean was generated externally (not computed here). Verify it matches  
the `hours > 12` threshold used by the pipeline. If they disagree, the overtime rate  
shown on the dashboard depends on which source we trust.

### 8.3 Date range interpretation
The staffing CSV covers 2022–2025. The dashboard does not currently filter to a specific  
date window. Metrics include all years equally. A high overtime year in 2022 affects  
the overall rate even if 2025 is healthy.

### 8.4 Shift encoding
Shift is encoded as plain text ('Day', 'Evening', 'Night'). No time range is defined  
for each label. If the agency changes shift names, or if entries use 'AM'/'PM' style,  
the by-shift breakdown will silently grow new categories.

### 8.5 No payroll linkage
This file does not contain pay rate, benefit cost, or compensation data.  
The payroll XLSX is a separate aggregate report (see notebook 03) with no join key.  
Cost-per-shift cannot be computed from this data.

### 8.6 What this data cannot tell you
- It cannot tell you why someone worked overtime (coverage gap vs. choice)
- It cannot tell you which unit each employee was assigned to on a given shift
- It cannot tell you whether a gap day was covered by mutual aid or left uncovered
- It cannot measure scheduling efficiency (planned vs. actual staffing)